<a href="https://colab.research.google.com/github/EssaouriYassine/MapReduce_paradigm/blob/main/ESSAOURI_ZEDDAOUI_TP_MapReduce_exo_1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Compte rendu du TD "Le Paradigme MapReduce" (1/4)

---

**Auteurs :** Yassine ESSAOURI, Oumaima ZEDDAOUI | **Encadrant :** Francois BASSAC

**Module :** Big Data | **Master 2 Cyber** | **Session :** 2026-2027 | **Rendu :** 28/09/2026 | **Statut :** Terminé

---

<br>

# Exercice 1 : Comptage de Mots (Word Count)

---

<br>

## 1. Objectif

L'objectif principal de cet exercice est de mettre en œuvre le paradigme de programmation distribuée **MapReduce** sur le problème classique du comptage de fréquence de mots (Word Count).

Cet exercice sert d'introduction fondamentale pour :
- Manipuler la logique d'extraction de paires $(\text{clé}, \text{valeur})$ dans la phase **Map**.
- Comprendre la phase intermédiaire de regroupement **Shuffle & Sort** exécutée par le framework.
- Effectuer des agrégations de données sur des ensembles de valeurs associés à une même clé dans la phase **Reduce**.

<br>

## 2. Problématique

- **Données d'entrée (Input) :** Un corpus de texte non structuré fourni sous forme de fichier texte (`Lorem Ipsum.txt`).
- **Données de sortie (Output) :** Une liste ou dictionnaire de paires $(\text{mot}, \text{fréquence})$ indiquant le nombre exact d'occurrences de chaque mot présent dans le corpus.
- **Contraintes & Défis :**
  - Traiter la casse (convertir en minuscules) et nettoyer la ponctuation pour éviter d'avoir des clés distinctes pour un même mot (ex: `"Lorem,"` vs `"lorem"`).
  - Respecter strictement le passage de messages au format $(\text{clé}, \text{valeur})$ propre à MapReduce.

<br>

## 3. Mise en forme des sorties

Cette section précise les conventions d'affichage et de style utilisées pour présenter les résultats, logs et retours de commandes de manière claire et homogène.

In [ ]:
import os
import re

from typing import List, Tuple, Dict

def print_separator(length: int = 50) -> None:
    # Affiche une ligne de séparation dans le terminal.
    print("-" * length)


def log_execution_start() -> None:
    # Affiche l'en-tête de lancement des commandes.
    print("▶ Lancement des commandes...\n")

def afficher_resultats(data, titre="RÉSULTAT", limite=10):
  # Normalisation en liste de paires (clé, valeur)
  if isinstance(data, dict):
    elements = list(data.items())
  else:
    elements = list(data)

  total = len(elements)
  reste = total - limite

  print(f"┌─ {titre.upper()} ─── ({total} éléments au total)")
  print("│")
  for i, (cle, valeur) in enumerate(elements[:limite], start=1):
    print(f"├─ {i:02d} — ({cle} , {valeur})")
  print("│")

  if reste > 0:
    print(f"└─ ... {reste} autres éléments masqués.")
  else:
    print(f"└─ Fin de la liste ({total} éléments).")

<br>

## 4. Démarche

La démarche suit les phases canoniques du modèle MapReduce :

### 4.1. **Préparation & Chargement des Données**
Lecture du fichier `Lorem Ipsum.txt`, nettoyage préliminaire (suppression de la ponctuation, passage en minuscules) et découpage en mots.

In [ ]:
print_separator()
log_execution_start()

# Chargement du fichier
nom_fichier = 'Lorem_Ipsum.txt'

if os.path.exists(nom_fichier):
    print(f"Chargement direct du fichier : {nom_fichier}")
    with open(nom_fichier, 'r', encoding='utf-8') as fichier:
        texte = fichier.read()
else:
    print(f"Fichier '{nom_fichier}' introuvable. Recherche d'une alternative...")
    fichiers = os.listdir('/content')
    fichier_trouve = next((f for f in fichiers if 'lorem' in f.lower()), None)

    if fichier_trouve is not None:
        print(f"Fichier trouvé : {fichier_trouve}")
        with open(f'/content/{fichier_trouve}', 'r', encoding='utf-8') as fichier:
            texte = fichier.read()
    else:
        raise FileNotFoundError("Aucun fichier Lorem Ipsum n'a été trouvé dans le répertoire.")

# Nettoyage : conversion en minuscules et extraction des mots
mots = re.findall(r'\b\w+\b', texte.lower())


# Vérification du résultat
msg = f"Fichier chargé avec succès ! Nombre total de mots : {len(mots)}"
print(msg)
print_separator(len(msg))

--------------------------------------------------
▶ Lancement des commandes...

Chargement direct du fichier : Lorem_Ipsum.txt
Fichier chargé avec succès ! Nombre total de mots : 670
-------------------------------------------------------


#### Explication
Le test `if / else` : Vérifie l'existence du fichier Lorem Ipsum.txt dans l'environnement. S'il n'est pas présent à la racine, le bloc else parcourt le répertoire pour localiser un fichier équivalent. Cela évite les blocages liés à des erreurs de nommage ou d'emplacement sur Colab.

`re.findall(r'\b\w+\b', texte.lower())` :

- `lower()` passe l'ensemble du texte en minuscules pour traiter de la même manière "Mot" et "mot".

- `re.findall` extrait uniquement les suites de caractères alphabétiques/numériques et ignore la ponctuation (points, virgules, espaces). On obtient une liste simple de mots : ['lorem', 'ipsum', 'dolor', ...].

<br>

### 4.2. **Phase Map :**

* **Rôle :** Transformer chaque mot de la liste en une paire `(clé, valeur)`.

* **Entrée :** Une liste de mots bruts.

* **Sortie :** Une liste de tuples `(mot, 1)`.

* **Fonctionnement :** La boucle parcourt chaque mot et associe la valeur 1 à chacun d'eux (ex: [('lorem', 1), ('ipsum', 1), ('lorem', 1)]). La fonction émet ces paires intermédiaires.

In [ ]:
def map_function(mots: List[str]) -> List[Tuple[str, int]]:

    paires: List[Tuple[str, int]] = []

    for mot in mots:
        paires.append((mot, 1))

    return paires

# Vérification du résultat
print_separator()
log_execution_start()

paires = map_function(mots)
afficher_resultats(paires, titre="Sortie Mapper", limite=10)
print_separator()

--------------------------------------------------
▶ Lancement des commandes...

┌─ SORTIE MAPPER ─── (670 éléments au total)
│
├─ 01 — (lorem , 1)
├─ 02 — (ipsum , 1)
├─ 03 — (dolor , 1)
├─ 04 — (sit , 1)
├─ 05 — (amet , 1)
├─ 06 — (consectetuer , 1)
├─ 07 — (adipiscing , 1)
├─ 08 — (elit , 1)
├─ 09 — (ut , 1)
├─ 10 — (purus , 1)
│
└─ ... 660 autres éléments masqués.
--------------------------------------------------


<br>

### 4.3. **Phase Shuffle & Sort (Simulée) :**

* **Rôle :** Regrouper toutes les occurrences associées à une même clé `(mot)`.  

* **Entrée :** La liste des paires intermédiaires `(mot, 1)`.

* **Sortie :** Un dictionnaire associant chaque mot à la liste de ses occurrences : `(mot, [1, 1, 1...])`.

* **Fonctionnement :** Si le mot n'a pas encore été rencontré `(if mot not in groupes)`, on crée une nouvelle clé dans le dictionnaire avec une liste vide. On ajoute ensuite le 1 à la liste correspondant à ce mot.

In [ ]:
def shuffle_and_sort(paires_map: List[Tuple[str, int]]) -> Dict[str, List[int]]:
    groupes = {}
    for mot, valeur in paires_map:
        if mot not in groupes:
            groupes[mot] = []
        groupes[mot].append(valeur)
    return groupes

# Vérification du résultat
print_separator()
log_execution_start()

groupes = shuffle_and_sort(paires)
afficher_resultats(groupes, titre="Shuffle & Sort", limite=10)
print_separator()

--------------------------------------------------
▶ Lancement des commandes...

┌─ SHUFFLE & SORT ─── (180 éléments au total)
│
├─ 01 — (lorem , [1, 1, 1, 1, 1, 1, 1])
├─ 02 — (ipsum , [1, 1, 1, 1, 1])
├─ 03 — (dolor , [1, 1, 1, 1, 1])
├─ 04 — (sit , [1, 1, 1, 1, 1, 1, 1])
├─ 05 — (amet , [1, 1, 1, 1, 1, 1, 1])
├─ 06 — (consectetuer , [1, 1, 1, 1])
├─ 07 — (adipiscing , [1, 1, 1, 1])
├─ 08 — (elit , [1, 1, 1, 1, 1, 1, 1, 1])
├─ 09 — (ut , [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1])
├─ 10 — (purus , [1, 1, 1, 1, 1, 1, 1, 1])
│
└─ ... 170 autres éléments masqués.
--------------------------------------------------


<br>

### 4.4. **Phase Reduce :**

* **Rôle :** Aggréger (sommer) les valeurs associées à chaque clé pour calculer la fréquence finale.

* **Entrée :** Le dictionnaire des groupes `(mot, [1, 1, 1...])`.

* **Sortie :** Un dictionnaire associant chaque mot à son total d'apparitions : `(mot, fréquence_total)`.

* **Fonctionnement :** Pour chaque mot, la fonction parcourt la liste de 1 associés, les additionne via une variable total, puis enregistre le résultat final.

In [ ]:
def reduce_function(groupes_mots: Dict[str, List[int]]) -> Dict[str, int]:
    resultat = {}
    for mot, liste_valeurs in groupes_mots.items():
        total = 0
        for valeur in liste_valeurs:
            total += valeur
        resultat[mot] = total
    return resultat

# Vérification du résultat
print_separator()
log_execution_start()

resultat_final = reduce_function(groupes)
afficher_resultats(resultat_final, titre="Résultat Reducer", limite=10)
print_separator()

--------------------------------------------------
▶ Lancement des commandes...

┌─ RÉSULTAT REDUCER ─── (180 éléments au total)
│
├─ 01 — (lorem , 7)
├─ 02 — (ipsum , 5)
├─ 03 — (dolor , 5)
├─ 04 — (sit , 7)
├─ 05 — (amet , 7)
├─ 06 — (consectetuer , 4)
├─ 07 — (adipiscing , 4)
├─ 08 — (elit , 8)
├─ 09 — (ut , 11)
├─ 10 — (purus , 8)
│
└─ ... 170 autres éléments masqués.
--------------------------------------------------


<br>

### 5. **Conclusion**

L'implémentation de cet exercice illustre le fonctionnement fondamental du paradigme **MapReduce** sur un problème de comptage de mots (Word Count) :   
- **Séparation des responsabilités :** La fonction **Map** transforme le texte brut en paires d'occurrences simples `(mot, 1)`, tandis que la fonction **Reduce** s'occupe uniquement d'agréger ces valeurs.

- **Rôle clé du Shuffle & Sort :** L'étape intermédiaire est essentielle car elle regroupe efficacement toutes les occurrences par clé `(mot, [1, 1, ...])`, permettant au Reducer de fonctionner de manière totalement indépendante pour chaque mot.   

<br>

---

<br>

### Exportation du notebook

Cette section présente la procédure d'automatisation et de conversion du notebook au format HTML via Google Colab et `nbconvert`.

In [ ]:
import os
from google.colab import files, drive

# 1. Montage du Drive
drive.mount('/content/drive', force_remount=True)

# 2. Nom du fichier à chercher
target_name = 'ESSAOURI-ZEDDAOUI-TP-MapReduce_exo_1.ipynb'
found_path = None

# Recherche récursive dans tout le Drive
print('Recherche du notebook dans votre Google Drive...')
for root, dirs, filenames in os.walk('/content/drive/MyDrive'):
  for filename in filenames:
    if filename.lower() == target_name.lower():
      found_path = os.path.join(root, filename)
      break
  if found_path:
    break

if found_path:
  print(f'✅ Fichier trouvé : {found_path}')
  # Conversion en HTML
  !jupyter nbconvert --to html "{found_path}"

  # Téléchargement automatique du fichier HTML généré
  html_name = os.path.basename(found_path).replace('.ipynb', '.html')
  if os.path.exists(html_name):
    files.download(html_name)
  else:
    # Si nbconvert génère le HTML dans le même dossier que le notebook
    html_in_drive = found_path.replace('.ipynb', '.html')
    if os.path.exists(html_in_drive):
      files.download(html_in_drive)
else:
  print(
      '❌ Fichier introuvable sur Google Drive. Vérifiez le nom exact du'
      ' fichier.'
  )

Mounted at /content/drive
Recherche du notebook dans votre Google Drive...
✅ Fichier trouvé : /content/drive/MyDrive/Colab Notebooks/ESSAOURI-ZEDDAOUI-TP-MapReduce_exo_1.ipynb
[NbConvertApp] Converting notebook /content/drive/MyDrive/Colab Notebooks/ESSAOURI-ZEDDAOUI-TP-MapReduce_exo_1.ipynb to html
[NbConvertApp] Writing 317484 bytes to /content/drive/MyDrive/Colab Notebooks/ESSAOURI-ZEDDAOUI-TP-MapReduce_exo_1.html


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>